# MaintQA 챗봇 Colab GPU 실습 — 정비 질문을 SQL로 바꾸는 LLM을 GPU에서 실행·평가

**박근태 · 20220281 · 오프라인 MaintQA 정비이력 챗봇 연계 실습**

내 PC에서 만든 MaintQA 챗봇은 질문을 받으면 **LLM(대규모 언어모델)** 이 그 질문을 **SQL**(DB에 보내는 조회 명령문)로 바꾸고, 그 SQL을 **DuckDB**에서 실행해 숫자를 그대로 보여 줍니다. 내 PC에서는 그래픽카드를 쓰지 못해 CPU로만 동작했기 때문에, 이 실습에서는 같은 과정을 **Colab의 T4 GPU**에서 실행해 정확도와 속도를 확인합니다.

| 항목 | 내용 |
|---|---|
| 모델 | `gemma4:e2b` — 챗봇에 쓴 것과 같은 모델, Ollama로 실행 |
| 프롬프트 | 챗봇 소스(`app/engine.py`)의 SQL 생성 프롬프트를 그대로 사용 |
| 데이터 | GitHub 저장소의 공개용 **가상** 정비기록 12건 (수업 제공 DB는 사용·업로드하지 않음) |
| 생성 설정 | temperature=0, top_k=1, seed=42 (매번 같은 답이 나오도록 고정) |
| 평가 | 정비 질문 12개 → 생성된 SQL의 실행 결과를 미리 만든 정답과 비교 |

실습 순서: ① GPU와 모델 준비 → ② 챗봇 소스·가상 DB 준비 → ③ 질문을 SQL로 바꿔 실행 → ④ 정답 비교·정확도 → ⑤ GPU와 CPU 속도 비교

## 01 · GPU 연결 확인과 LLM 실행 환경(Ollama) 준비

In [ ]:
import subprocess, time, json, os, requests

print("증빙 01 | GPU 연결 및 모델 준비")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "GPU가 없습니다: 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 다시 실행하세요."
print("GPU:", gpu)

# Ollama: 내 PC에서 쓰던 것과 같은 방식으로 로컬 LLM을 실행해 주는 프로그램
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
OLLAMA = "http://127.0.0.1:11434"
for _ in range(60):
    try:
        requests.get(OLLAMA, timeout=1); break
    except Exception:
        time.sleep(1)
print("Ollama 버전:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

MODEL = "gemma4:e2b"
t0 = time.time()
subprocess.run(["ollama", "pull", MODEL], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"모델 다운로드: {MODEL} ({time.time()-t0:.0f}초)")

# 짧은 질문으로 모델을 GPU에 올린 뒤, 어느 장치에서 돌고 있는지 확인
requests.post(f"{OLLAMA}/api/generate", json={"model": MODEL, "prompt": "안녕", "stream": False,
              "options": {"num_predict": 5}}, timeout=600)
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 02 · 챗봇 소스와 공개용 가상 정비 DB 준비

In [ ]:
import sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "koreanize-matplotlib"], check=True)
import duckdb, pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프 한글 표시

REPO = "/content/maintqa-colab-submission"
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "https://github.com/gguntt/maintqa-colab-submission.git", REPO], check=True)
APP = f"{REPO}/app"
subprocess.run([sys.executable, "create_public_demo.py"], cwd=APP, check=True)
sys.path.insert(0, APP)
import config, database, engine, router   # 챗봇 소스 모듈을 그대로 불러옴

settings = config.read()
con = database.connect(settings)          # 챗봇과 같은 '읽기 전용' 연결
tables = [t[0] for t in con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").fetchall()]
counts = {t: con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0] for t in tables}

print("증빙 02 | 챗봇 소스와 가상 DB 준비")
print("데이터 출처:", settings["dataset_notice"])
print("DuckDB 버전:", duckdb.__version__, "| 연결 방식: 읽기 전용(read_only=True)")
print(pd.DataFrame({"테이블": list(counts), "행 수": list(counts.values())}).to_string(index=False))

names = {"master": "정비 문서", "pms": "작업 라인", "vessels": "선박", "approval": "결재",
         "attachments": "첨부", "consumption": "부품 소모"}
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.bar([f"{names.get(t, t)}\n({t})" for t in counts], counts.values(), color="#4C72B0")
ax.bar_label(bars)
ax.set(title="공개용 가상 정비 DB의 테이블별 기록 수", ylabel="행 수")
fig.tight_layout(); plt.show()

## 03 · 챗봇 프롬프트로 질문 → SQL 생성 → DuckDB 실행 (GPU)

In [ ]:
import threading
# 정비 질문 12개와, 사람이 직접 작성한 정답 SQL
QUESTIONS = [
    ("전체 정비 문서는 몇 건인가요?", "SELECT COUNT(*) FROM master"),
    ("선박은 총 몇 척인가요?", "SELECT COUNT(*) FROM vessels"),
    ("작업 라인은 모두 몇 건인가요?", "SELECT COUNT(*) FROM pms"),
    ("선박별 문서 개수를 알려 주세요", "SELECT VESSELCODE, COUNT(*) FROM master GROUP BY VESSELCODE"),
    ("훈련용 펌프 정비 문서는 몇 건인가요?", "SELECT COUNT(*) FROM master WHERE EQUIP_DESC='TRAINING PUMP'"),
    ("훈련용 팬 문서 건수는?", "SELECT COUNT(*) FROM master WHERE EQUIP_DESC='TRAINING FAN'"),
    ("부품 소모 기록이 있는 문서는 몇 건인가요?", "SELECT COUNT(*) FROM (SELECT DISTINCT VESSELCODE, DOC_NO FROM consumption)"),
    ("부품 소모 수량의 합계는?", "SELECT SUM(CONS_QTY) FROM consumption"),
    ("DEMO03 선박의 문서 목록을 알려 주세요", "SELECT DOC_NO FROM master WHERE VESSELCODE='DEMO03'"),
    ("결재 상태가 DEMO_COMPLETE인 문서는 몇 건인가요?", "SELECT COUNT(*) FROM approval WHERE STATUS='DEMO_COMPLETE'"),
    ("첨부파일은 몇 건인가요?", "SELECT COUNT(*) FROM attachments"),
    ("문서번호 D001인 문서는 몇 건인가요?", "SELECT COUNT(*) FROM master WHERE DOC_NO='D001'"),
]
prefix = engine.aggregate_prefix(settings, con)   # 챗봇이 쓰는 지시문 + 테이블 구조 + 규칙 + 예시

def ask_llm(prompt, gpu=True):
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}], "stream": False, "think": False,
            "options": {"temperature": 0, "top_k": 1, "seed": 42, "num_predict": 600}}
    if not gpu: body["options"]["num_gpu"] = 0          # 0이면 GPU를 끄고 CPU로만 계산
    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=1200)
    if r.status_code == 400:                              # 'think' 옵션을 지원하지 않는 버전 대비
        body.pop("think"); r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=1200)
    r.raise_for_status(); d = r.json()
    return d["message"]["content"], d

def clean(sql):                                          # 챗봇과 같은 방식으로 ``` 표시 제거
    sql = sql.strip()
    if sql.startswith("```"): sql = sql.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return sql

def run_sql(sql):
    try:
        return database.query(con, sql, threading.Event())["rows"], ""
    except Exception as e:
        return None, str(e)[:80]

print("증빙 03 | 질문 → SQL 생성 → DuckDB 실행 (T4 GPU)")
print("질문 분류(router.py):", {q: router.route(q) for q, _ in QUESTIONS[:2]}, "...")
records = []
for i, (q, gold_sql) in enumerate(QUESTIONS, 1):
    t = time.perf_counter()
    text, meta = ask_llm(prefix + q, gpu=True)
    sec = time.perf_counter() - t
    sql = clean(text)
    rows, err = run_sql(sql)
    gold, _ = run_sql(gold_sql)
    records.append(dict(no=i, question=q, route=router.route(q), sql=sql, rows=rows, error=err, gold=gold,
                        seconds=sec, tokens=meta.get("eval_count", 0),
                        tok_per_s=meta.get("eval_count", 0) / max(meta.get("eval_duration", 1) / 1e9, 1e-9)))
    shown = (str(rows)[:60] if rows is not None else "실행 실패: " + err)
    print(f"Q{i:02d} {q}\n     SQL: {sql[:110]}\n     결과: {shown} | {sec:.1f}초")

## 04 · 정답과 비교한 정확도 평가 및 시각화

생성된 SQL의 **실행 결과**가 정답 SQL의 결과와 같으면 정답으로 봅니다. (SQL 문장 모양이 달라도 결과가 같으면 정답)

In [ ]:
def norm(rows):
    if rows is None: return None
    out = []
    for r in rows:
        vals = []
        for v in r:
            try: vals.append(str(float(v)).rstrip("0").rstrip("."))
            except (TypeError, ValueError): vals.append(str(v))
        out.append(tuple(sorted(vals)))
    return sorted(out)

for r in records:
    r["correct"] = r["rows"] is not None and norm(r["rows"]) == norm(r["gold"])
    r["status"] = "정답" if r["correct"] else ("실행 실패" if r["rows"] is None else "오답")
df = pd.DataFrame(records)
acc = df["correct"].mean()

print("증빙 04 | 정답 비교 평가")
print(df[["no", "question", "status", "seconds"]].rename(columns={"no": "번호", "question": "질문", "status": "판정", "seconds": "응답(초)"})
        .round(2).to_string(index=False))
print(f"\n정확도: {df['correct'].sum()}/{len(df)} = {acc:.1%}")

colors = {"정답": "#55A868", "오답": "#DD8452", "실행 실패": "#C44E52"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [3, 1.2]})
axes[0].barh([f"Q{n:02d} {q[:14]}" for n, q in zip(df.no, df.question)], 1,
             color=[colors[s] for s in df.status])
axes[0].invert_yaxis(); axes[0].set_xticks([]); axes[0].set_title("질문별 판정 (초록=정답)")
cnt = df.status.value_counts()
axes[1].pie(cnt.values, labels=cnt.index, autopct="%d%%", colors=[colors[k] for k in cnt.index], startangle=90)
axes[1].set_title(f"전체 정확도 {acc:.0%}")
fig.tight_layout(); plt.show()

## 05 · GPU와 CPU 속도 비교 및 결과 저장

In [ ]:
print("증빙 05 | GPU vs CPU 속도 비교")
sample = QUESTIONS[:3]                     # CPU는 느려서 3개 질문만 비교
cpu = []
for q, _ in sample:
    t = time.perf_counter(); _, meta = ask_llm(prefix + q, gpu=False); sec = time.perf_counter() - t
    cpu.append(dict(question=q, seconds=sec,
                    tok_per_s=meta.get("eval_count", 0) / max(meta.get("eval_duration", 1) / 1e9, 1e-9)))
gpu_s = df.loc[:2, "seconds"].mean(); cpu_s = pd.DataFrame(cpu)["seconds"].mean()
gpu_tps = df.loc[:2, "tok_per_s"].mean(); cpu_tps = pd.DataFrame(cpu)["tok_per_s"].mean()
print(f"같은 질문 3개 평균 응답 시간 — GPU {gpu_s:.1f}초 / CPU {cpu_s:.1f}초 → GPU가 약 {cpu_s/gpu_s:.1f}배 빠름")
print(f"초당 생성 토큰(글자 조각) — GPU {gpu_tps:.1f}개 / CPU {cpu_tps:.1f}개")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
b = axes[0].bar(["GPU (T4)", "CPU"], [gpu_s, cpu_s], color=["#4C72B0", "#999999"]); axes[0].bar_label(b, fmt="%.1f초")
axes[0].set(title="질문당 평균 응답 시간", ylabel="초")
b = axes[1].bar(["GPU (T4)", "CPU"], [gpu_tps, cpu_tps], color=["#4C72B0", "#999999"]); axes[1].bar_label(b, fmt="%.1f")
axes[1].set(title="초당 생성 토큰 수", ylabel="토큰/초")
axes[2].bar([f"Q{n:02d}" for n in df.no], df.seconds, color=[colors[s] for s in df.status])
axes[2].set(title="GPU 질문별 응답 시간 (색=판정)", ylabel="초")
fig.tight_layout(); plt.show()

OUT = "/content/maintqa_colab_results"; os.makedirs(OUT, exist_ok=True)
df.drop(columns=["rows", "gold"]).to_csv(f"{OUT}/sql_results.csv", index=False, encoding="utf-8-sig")
json.dump({"model": MODEL, "gpu": gpu, "accuracy": acc, "n_questions": len(df),
           "gpu_avg_seconds": gpu_s, "cpu_avg_seconds": cpu_s, "gpu_tok_per_s": gpu_tps, "cpu_tok_per_s": cpu_tps},
          open(f"{OUT}/summary.json", "w"), ensure_ascii=False, indent=2)
print("결과 저장:", OUT, os.listdir(OUT))